# Importando as bibliotecas

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from pyspark.sql import functions as F
from pyspark.sql import Window as W

# Criando o database silver

In [0]:
%sql
USE CATALOG projeto;
-- DROP DATABASE silver CASCADE; -- Executar se tiver uma silver já criada
CREATE DATABASE IF NOT EXISTS silver;

# bronze.chamados_hora -> silver.dim_chamado_hora

In [0]:
# Lendo a tabela chamados_hora da camada bronze e visualizando os primeiros registros
df_bz = spark.table("bronze.chamados_hora")
print(f"bronze.chamados_hora: {df_bz.count()} rows")
display(df_bz.limit(10))

In [0]:
# Renomeando as colunas para snake_case
df = (
    df_bz
    .withColumnRenamed("ID_Chamado", "id_chamado")
    .withColumnRenamed("ID_Cliente", "id_cliente")
    .withColumnRenamed("Hora_Abertura_Chamado", "hora_abertura_chamado_raw")
    .withColumnRenamed("Hora_Inicio_Atendimento", "hora_inicio_atendimento_raw")
    .withColumnRenamed("Hora_Finalizacao_Atendimento", "hora_finalizacao_atendimento_raw")
    # ingestion_timestamp já está em snake_case
)

# Criando uma função para tirar o " �s " e converter pra timestamp
def to_ts(col):
    return F.to_timestamp(F.regexp_replace(col, " �s ", " "), "dd/MM/yyyy HH:mm:ss")

df = (
    # Aplicando a função nas três colunas de tempo e dropando as raws
    df
    .withColumn("data_hora_abertura", to_ts(F.col("hora_abertura_chamado_raw")))
    .withColumn("data_hora_inicio_atendimento", to_ts(F.col("hora_inicio_atendimento_raw")))
    .withColumn("data_hora_finalizacao_atendimento", to_ts(F.col("hora_finalizacao_atendimento_raw")))
    .drop(
        "hora_abertura_chamado_raw",
        "hora_inicio_atendimento_raw",
        "hora_finalizacao_atendimento_raw"
    )

    # garantir tipos dos IDs em long
    .withColumn("id_chamado", F.col("id_chamado").cast("long"))
    .withColumn("id_cliente", F.col("id_cliente").cast("long"))

    # Criando duas métricas úteis de minutos
    .withColumn(
        "tempo_espera_atendimento_min",
        F.round((F.col("data_hora_inicio_atendimento").cast("long") - F.col("data_hora_abertura").cast("long")) / 60.0, 2)
    )

    .withColumn(
        "tempo_atendimento_min",
        F.round(
            (F.col("data_hora_finalizacao_atendimento").cast("long") - F.col("data_hora_inicio_atendimento").cast("long")) / 60.0, 2)
    )
)

# Ordenando as colunas
df = df.select(
    "id_chamado",
    "id_cliente",
    "data_hora_abertura",
    "data_hora_inicio_atendimento",
    "data_hora_finalizacao_atendimento",
    "tempo_espera_atendimento_min",
    "tempo_atendimento_min",
    "ingestion_timestamp"
)

display(df.limit(10))


In [0]:
# Salvando no silver.dim_chamado_hora (formato delta por padrão)
df.write.mode("overwrite").saveAsTable("silver.dim_chamado_hora")
print(f"silver.dim_chamado_hora: {df.count()} rows")